# CEMS API ingestion

## Executive summary
Read one CEMS activation from the Montandon STAC catalog (`cems-events`, `cems-hazards`, `cems-response`, `cems-impacts`). GeoJSON assets on response items are cached under `data/cache/cems/{activation_id}/`. Notebooks 02 and 03 read that cache.

## Prerequisites
Install `requirements.txt`. Set `MONTANDON_STAC_TOKEN` in the environment (bearer token from the IFRC identity provider). The activation ID comes from `config/params.yaml`. ZIP packages, COG imagery, and SLD styles are listed and not downloaded.


## Parameters

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.copernicus_loader import load_params

params = load_params(PROJECT_ROOT)
ACTIVATION_ID = params['activation_id']
print({
    'activation_id': ACTIVATION_ID,
    'hazard_type': params.get('hazard_type'),
    'country_code': params.get('country_code'),
})

{'activation_id': 'EMSR884', 'hazard_type': 'earthquake', 'country_code': 'VEN'}


## Event
One authenticated call loads `cems-event-{activation_id}`. Related hazards, responses, and impacts are then searched by `monty:corr_id`.


In [2]:
from src.cems_api import event_summary, fetch_stac_activation

activation = fetch_stac_activation(ACTIVATION_ID)
print(event_summary(activation['event']))
print({
    'hazards': len(activation['hazards']),
    'responses': len(activation['responses']),
    'impacts': len(activation['impacts']),
})


{'id': 'cems-event-EMSR884', 'title': 'Earthquake in Venezuela', 'datetime': '2026-06-24T16:04:00Z', 'country_codes': 'VEN', 'hazard_codes': 'GH0101, EQ, nat-geo-ear-gro', 'corr_id': '20260624-VEN-902362-GH0101-1-GCDB'}
{'hazards': 13, 'responses': 45, 'impacts': 42}


## Hazards, responses, and assets
Response rows come from `monty:response_detail`. Only assets whose media type is `application/geo+json` are marked for download.


In [3]:
from src.cems_api import asset_index, hazard_table, response_table

display(hazard_table(activation['hazards']))
display(response_table(activation['responses']))
display(asset_index(activation['responses']))


,id,title,datetime,hazard_codes,geometry_type
0,cems-hazard-EMSR884-aoi12-earthquake,Caraballeda — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
1,cems-hazard-EMSR884-aoi11-earthquake,Villa de Cura — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
2,cems-hazard-EMSR884-aoi10-earthquake,Guacara — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
3,cems-hazard-EMSR884-aoi09-earthquake,Valencia — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
4,cems-hazard-EMSR884-aoi08-earthquake,San Felipe — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
5,cems-hazard-EMSR884-aoi07-earthquake,Puerto Cabello — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
6,cems-hazard-EMSR884-aoi06-earthquake,Moron — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
7,cems-hazard-EMSR884-aoi05-earthquake,Santa Cruz — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
8,cems-hazard-EMSR884-aoi04-earthquake,Maracay — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon
9,cems-hazard-EMSR884-aoi03-earthquake,Antimano — ground shaking hazard area,2026-06-24T16:04:00Z,"GH0101, EQ, nat-geo-ear-gro",Polygon


,id,title,datetime,response_type,status,monitoring_number,source_id
0,cems-response-EMSR884-aoi00-gra,CEMS grading product — Central Coastal Venezue...,2026-07-06T14:42:15.412962Z,eo-gra,published,NaN,EMSR884
1,cems-response-EMSR884-aoi12-gra-m2,CEMS grading product — Caraballeda (EMSR884),2026-07-06T07:46:29.908321Z,eo-gra,published,2.0,EMSR884
2,cems-response-EMSR884-aoi08-gra-m2,CEMS grading product — San Felipe (EMSR884),2026-07-06T00:48:00Z,eo-gra,no-impact,2.0,EMSR884
3,cems-response-EMSR884-aoi08-dat-worldview2_202...,CEMS optical acquisition — Legion,2026-07-05T15:17:00Z,eo-dat,NaN,NaN,EMSR884
4,cems-response-EMSR884-aoi12-dat-geoeye1_202607...,CEMS optical acquisition — GeoEye-1,2026-07-05T15:05:00Z,eo-dat,NaN,NaN,EMSR884
5,cems-response-EMSR884-aoi12-gra-m1,CEMS grading product — Caraballeda (EMSR884),2026-07-02T04:12:15.565161Z,eo-gra,published,1.0,EMSR884
6,cems-response-EMSR884-aoi02-gra-m1,CEMS grading product — Caracas (EMSR884),2026-06-29T18:18:07.700477Z,eo-gra,published,1.0,EMSR884
7,cems-response-EMSR884-aoi08-gra-m1,CEMS grading product — San Felipe (EMSR884),2026-06-29T17:28:21.981067Z,eo-gra,published,1.0,EMSR884
8,cems-response-EMSR884-aoi06-gra-m1,CEMS grading product — Moron (EMSR884),2026-06-29T11:25:23.016205Z,eo-gra,published,1.0,EMSR884
9,cems-response-EMSR884-aoi04-gra,CEMS grading product — Maracay (EMSR884),2026-06-28T17:46:00Z,eo-gra,withdrawn,NaN,EMSR884


,item_id,asset,media_type,roles,href,download
0,cems-response-EMSR884-aoi00-gra,download,application/zip,data,https://rapidmapping.emergency.copernicus.eu/b...,False
1,cems-response-EMSR884-aoi00-gra,notAnalysedA,application/geo+json,data,https://rapidmapping-viewer.s3.eu-west-1.amazo...,True
2,cems-response-EMSR884-aoi00-gra,groundMovementA,application/geo+json,data,https://rapidmapping-viewer.s3.eu-west-1.amazo...,True
3,cems-response-EMSR884-aoi00-gra,notAnalysedA_style,application/vnd.ogc.sld+xml,metadata,https://rapidmapping-viewer.s3.eu-west-1.amazo...,False
4,cems-response-EMSR884-aoi00-gra,groundMovementA_style,application/vnd.ogc.sld+xml,metadata,https://rapidmapping-viewer.s3.eu-west-1.amazo...,False
...,...,...,...,...,...,...
130,cems-response-EMSR884-aoi03-dat-legion_2026062...,data,image/tiff; application=geotiff; profile=cloud...,"data, visual",https://rapidmapping-viewer.s3.eu-west-1.amazo...,False
131,cems-response-EMSR884-aoi08-dat-pneo_20260625_...,data,image/tiff; application=geotiff; profile=cloud...,"data, visual",https://rapidmapping-viewer.s3.eu-west-1.amazo...,False
132,cems-response-EMSR884-aoi10-dat-pneo_20260625_...,data,image/tiff; application=geotiff; profile=cloud...,"data, visual",https://rapidmapping-viewer.s3.eu-west-1.amazo...,False
133,cems-response-EMSR884-aoi02-dat-pneo_20260625_...,data,image/tiff; application=geotiff; profile=cloud...,"data, visual",https://rapidmapping-viewer.s3.eu-west-1.amazo...,False


## Response vector layers
GeoJSON assets are saved under `data/cache/cems/{activation_id}/`, with `manifest.json` recording each STAC `datetime`. A file already in that folder is reused. One failed layer is printed and the rest continue.


In [4]:
from src.cems_api import load_response_vectors
from src.copernicus_loader import layer_summary

vector_layers, vector_errors = load_response_vectors(activation['responses'], ACTIVATION_ID)
if vector_errors:
    print('Layer errors:')
    for message in vector_errors:
        print(message)
print({'vector_layers': len(vector_layers), 'failed_layers': len(vector_errors)})

if vector_layers:
    display(layer_summary(vector_layers))
    for name, layer in vector_layers.items():
        preview_cols = [column for column in layer.columns if column != 'geometry'][:8]
        print(name)
        display(layer[preview_cols].head())


{'vector_layers': 40, 'failed_layers': 0}


,layer,feature_count,geometry_type,crs,columns
0,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,245,Polygon,EPSG:4326,"obj_type, or_src_id, cems_layer, stac_item, mo..."
1,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,615,"MultiPolygon, Polygon",EPSG:4326,"obj_desc, value, det_method, dmg_src_id, cems_..."
2,EMSR884_AOI12_GRA_MONIT02_builtUpP_v1,2817,Point,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
3,EMSR884_AOI12_GRA_MONIT02_facilitiesA_v1,6,Polygon,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
4,EMSR884_AOI12_GRA_MONIT02_notAnalysedA_v1,1,MultiPolygon,EPSG:4326,"obj_type, or_src_id, cems_layer, stac_item, mo..."
5,EMSR884_AOI12_GRA_MONIT02_transportationA_v1,1,Polygon,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
6,EMSR884_AOI12_GRA_MONIT02_transportationL_v1,3661,"LineString, MultiLineString",EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
7,EMSR884_AOI12_GRA_MONIT02_ancillaryCrisisInfoA_v1,12,Polygon,EPSG:4326,"obj_type, obj_desc, elem_count, det_method, no..."
8,EMSR884_AOI12_GRA_MONIT02_ancillaryCrisisInfoP_v1,76,Point,EPSG:4326,"obj_type, obj_desc, elem_count, det_method, no..."
9,EMSR884_AOI12_GRA_MONIT01_builtUpP_v3,2770,Point,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."


EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2


,obj_type,or_src_id,cems_layer,stac_item,monitoring_number,delivery_time
0,Not Analysed,997,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
1,Not Analysed,997,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
2,Not Analysed,997,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
3,Not Analysed,997,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
4,Not Analysed,997,EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z


EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2


,obj_desc,value,det_method,dmg_src_id,cems_layer,stac_item,monitoring_number,delivery_time
0,LOS Displacement,0.05 to 0.1,Automatic extraction,997,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
1,LOS Displacement,-0.05 to 0,Automatic extraction,997,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
2,LOS Displacement,-0.05 to 0,Automatic extraction,997,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
3,LOS Displacement,-0.05 to 0,Automatic extraction,997,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z
4,LOS Displacement,-0.05 to 0,Automatic extraction,997,EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2,cems-response-EMSR884-aoi00-gra,None,2026-07-06T14:42:15.412962Z


... (72 outputs left out)

EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1


,obj_type,or_src_id,cems_layer,stac_item,monitoring_number,delivery_time
0,Not Analysed,2,EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1,cems-response-EMSR884-aoi02-gra,None,2026-06-26T04:01:10.948274Z
1,Not Analysed,2,EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1,cems-response-EMSR884-aoi02-gra,None,2026-06-26T04:01:10.948274Z
2,Not Analysed,2,EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1,cems-response-EMSR884-aoi02-gra,None,2026-06-26T04:01:10.948274Z
3,Not Analysed,2,EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1,cems-response-EMSR884-aoi02-gra,None,2026-06-26T04:01:10.948274Z
4,Not Analysed,2,EMSR884_AOI02_GRA_PRODUCT_notAnalysedA_v1,cems-response-EMSR884-aoi02-gra,None,2026-06-26T04:01:10.948274Z


EMSR884_AOI02_GRA_PRODUCT_transportationL_v1


,obj_type,name,info,simplified,damage_gra,det_method,notation,or_src_id
0,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
1,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
2,"211-Highways, Streets and Roads",Calle San Antonio,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
3,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
4,"211-Highways, Streets and Roads",Avenida Intercomunal Hoyo de La Puerta - Baruta,21120-Primary Road,Main roads,No visible damage,Photo-interpretation,Not Applicable,994


## Apply the damage filter used downstream
Same rule as the local notebook: keep rows where `damage_column` equals `damage_positive_value`. Official CEMS layers usually have no `damage` column (that column belongs to the local Overture file). When it is missing, every geometry is kept and the attribute names are printed.

In [5]:
from src.copernicus_loader import bbox_from_geodataframe, concat_layers, filter_impact

if not vector_layers:
    print('No published vector layers were loaded, so the damage filter was not applied.')
else:
    impact_all = concat_layers(vector_layers)
    damage_column = (params.get('impact') or {}).get('damage_column', 'damage')
    impact = filter_impact(impact_all, params)
    buffer_deg = float((params.get('impact') or {}).get('bbox_buffer_deg', 0.05))
    print({
        'features_loaded': len(impact_all),
        'features_after_damage_filter': len(impact),
        'damage_column': damage_column,
        'damage_column_present': damage_column in impact_all.columns,
        'columns': [column for column in impact_all.columns if column != 'geometry'],
        'bounds_wgs84': impact.to_crs('EPSG:4326').total_bounds.tolist() if not impact.empty else None,
        'bbox_with_buffer': bbox_from_geodataframe(impact, buffer_deg) if not impact.empty else None,
    })

{'features_loaded': 53708, 'features_after_damage_filter': 6260, 'damage_column': 'damage_gra', 'damage_column_present': True, 'columns': ['obj_type', 'or_src_id', 'cems_layer', 'stac_item', 'monitoring_number', 'delivery_time', 'copernicus_layer', 'obj_desc', 'value', 'det_method', 'dmg_src_id', 'name', 'info', 'simplified', 'damage_gra', 'notation', 'cd_value', 'elem_count', 'real'], 'bounds_wgs84': [-68.85519831, 10.21239458, -66.55240553, 10.62701385], 'bbox_with_buffer': [-68.90519831, 10.162394579999999, -66.50240553, 10.677013850000002]}


## Montandon impact items
Figures already published on `cems-impacts`. These are Montandon totals, not the administrative overlay from notebook 02.


In [6]:
from src.cems_api import impact_stats_table

stats = impact_stats_table(activation['impacts'])
display(stats)


,item_id,datetime,category,type,value,unit,estimate_type
0,cems-impact-EMSR884-aoi00-gra-total,2026-07-06T14:42:15.412962Z,total_affected,affected_total,6415100.0,NaN,primary
1,cems-impact-EMSR884-aoi00-gra-0-m-to-0.05-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,230000.0,NaN,primary
2,cems-impact-EMSR884-aoi00-gra-0.5-m-to-0.2-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,200.0,NaN,primary
3,cems-impact-EMSR884-aoi00-gra->-0.5-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,0.0,NaN,primary
4,cems-impact-EMSR884-aoi00-gra-<-0.5-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,0.0,NaN,primary
5,cems-impact-EMSR884-aoi00-gra-0.2-m-to-0.5-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,3800000.0,NaN,primary
6,cems-impact-EMSR884-aoi00-gra-0.2-m-to-0.1-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,4900.0,NaN,primary
7,cems-impact-EMSR884-aoi00-gra-0.1-m-to-0.2-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,1900000.0,NaN,primary
8,cems-impact-EMSR884-aoi00-gra-0.1-m-to-0.05-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,50000.0,NaN,primary
9,cems-impact-EMSR884-aoi00-gra-0.05-m-to-0-m,2026-07-06T14:42:15.412962Z,total_affected,affected_total,20000.0,NaN,primary
